# 04: Piotroski F-Score (protótipo)

Este é o protótipo de onde saiu a tool `calcular_piotroski_score`. Ele explica o conceito, mostra a função de dados usada na época (`buscar_dados_acao_b3`) e calcula os 9 critérios com dados reais do yfinance. No fim, a seção "O que virou no projeto" conta o que mudou até a versão atual.

---

## O que é o Piotroski F-Score

**Joseph Piotroski** criou em 2000 um sistema simples para separar empresas financeiramente saudáveis das frágeis. É uma nota de **0 a 9**: cada critério vale 1 ponto se a empresa atende e 0 se não atende.

**Faixas usadas neste protótipo** (o projeto mudou depois, veja o fim do notebook):
- **8-9 pontos**: fundamentos sólidos (lucro, caixa e crescimento consistentes)
- **6-7 pontos**: intermediário, empresa razoável com algumas fraquezas
- **0-5 pontos**: fraca, com prejuízo, caixa ruim ou deterioração

**Os 9 critérios** se dividem em 3 grupos:

| Grupo | Critérios |
|------|-----------|
| **Lucratividade** | Net Income > 0, OCF > 0, Net Income crescente, OCF > NI |
| **Alavancagem e liquidez** | Ativo total crescente, passivo circulante caindo, liquidez corrente subindo |
| **Eficiência operacional** | Sem diluição de ações, margem bruta crescente |

**Por que importa?** No estudo original, empresas com Piotroski alto tiveram retorno melhor na bolsa. O score ajuda a separar quem lucra de verdade, com caixa, de quem só parece lucrar no papel.

### Um exemplo para ler o score

Imagine que a PETR4 tirasse 4/9 com este resultado:
- Net Income positivo e OCF positivo [OK]
- OCF > Net Income, ou seja, lucro com qualidade [OK]
- Sem diluição de ações [OK]
- Lucro caiu em relação ao trimestre anterior [NAO]
- Ativo total diminuiu [NAO]
- Passivo circulante aumentou [NAO]
- Liquidez piorou [NAO]
- Margem bruta encolheu [NAO]

A leitura seria: a empresa está lucrando, mas quase todos os indicadores estão piorando, o que é um sinal de alerta.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

In [2]:
# ── Secao 1: Setup ──

import os
import yfinance as yf
import pandas as pd


print("[OK] Dependencias carregadas (yfinance, pandas, dotenv)")

[OK] Dependencias carregadas (yfinance, pandas, dotenv)


In [3]:
# ── Secao 2: Ferramenta de Dados Financeiros ──
# Esta funcao vem de fluxo_financeiro/tools/financial_data.py
# Aqui a mostramos completa para voce entender como funciona.

def buscar_dados_acao_b3(ticker: str) -> str:
    """
    Busca os dados financeiros atuais e o historico recente de uma acao na B3.
    
    Parametros:
        ticker (str): Codigo da acao (ex: "PETR4" ou "PETR4.SA")
    
    Retorno:
        str: String formatada com preco atual e historico de 6 meses
    
    CONTEXTO: A B3 usa sufixo ".SA" para todas as acoes brasileiras.
    Esta funcao auto-adiciona se voce esquecer.
    """
    if not ticker.endswith('.SA'):
        ticker += '.SA'
    
    try:
        acao = yf.Ticker(ticker)
        hist = acao.history(period="6mo")  # Ultimo semestre
        
        if hist.empty:
            return f"Nao foram encontrados dados para o ticker {ticker}."
        
        info = acao.info
        preco_atual = info.get('currentPrice', hist['Close'].iloc[-1])
        
        resultado = f"Dados para {ticker}:\n"
        resultado += f"Preco Atual: R$ {preco_atual}\n"
        resultado += f"Historico Recente (Ultimos 5 dias de fechamento):\n{hist['Close'].tail(5).to_string()}\n"
        return resultado
    except Exception as e:
        return f"Erro ao buscar dados do ticker {ticker}: {e}"

print("[OK] Funcao buscar_dados_acao_b3 definida")

[OK] Funcao buscar_dados_acao_b3 definida


In [4]:
# ── Secao 3: Teste da ferramenta + Extracao de metricas basicas ──

ticker = 'PETR4.SA'

acao = yf.Ticker(ticker)
historico = acao.history(period="6mo")

print(f"Ticker: {ticker}")
print(f"Ultimas 3 datas de fechamento:")
print(historico['Close'].tail(3).to_string())
print(f"\nDados disponiveis em acao.info: {len(acao.info)} campos")

Ticker: PETR4.SA
Ultimas 3 datas de fechamento:
Date
2026-11-04 00:00:00-03:00    47.119999
2026-11-05 00:00:00-03:00    45.950001
2026-11-06 00:00:00-03:00    45.360001



Dados disponiveis em acao.info: 165 campos


## Os 9 critérios, um por um

### Lucratividade (4 critérios)

#### 1. Net Income > 0
- A empresa lucrou no trimestre?
- Se sim: +1 ponto

#### 2. Operating Cash Flow > 0
- A empresa transformou o lucro em dinheiro no caixa?
- Lucro só no papel (valores a receber) não conta, ele precisa virar caixa
- Se sim: +1 ponto
- Dados: `quarterly_cashflow['Operating Cash Flow']`

#### 3. Net Income crescente
- O lucro está crescendo de um trimestre para o outro?
- A pergunta é se a empresa está lucrando mais, e não só se lucrou
- Compara o trimestre atual com o anterior
- Se cresceu: +1 ponto

#### 4. Qualidade do lucro (OCF > Net Income)
- OCF maior que o lucro: o lucro está apoiado em caixa de verdade
- OCF menor que o lucro: parte do lucro pode ser só valores a receber, o que pede cuidado
- Se OCF > NI: +1 ponto

---

### Alavancagem e liquidez (3 critérios)

#### 5. Ativo total crescente
- A empresa está reinvestindo o lucro ou captando capital?
- Ativo crescendo pode ser bom (expansão) ou ruim (endividamento demais)
- Se o ativo aumentou: +1 ponto
- Dados: `quarterly_balance_sheet['Total Assets']`

#### 6. Passivo circulante caindo
- A empresa está reduzindo o que deve no curto prazo?
- Se diminuiu: +1 ponto
- Dados: `quarterly_balance_sheet['Current Liabilities']`

#### 7. Liquidez corrente subindo
- Liquidez corrente (Current Ratio) = ativo circulante / passivo circulante
- Mede a capacidade de pagar as contas de curto prazo
- Se aumentou: +1 ponto
- Dados: `quarterly_balance_sheet['Current Assets']` e `['Current Liabilities']`

---

### Eficiência operacional (2 critérios)

#### 8. Sem diluição de ações
- A empresa emitiu novas ações no trimestre?
- Diluição diminui o pedaço da empresa que cada ação representa
- Se o número de ações se manteve ou caiu: +1 ponto
- Dados: `quarterly_balance_sheet['Ordinary Shares Number']`

#### 9. Margem bruta crescente
- Margem bruta = (receita - custo do que foi vendido) / receita
- Mede quanto de cada real vendido sobra antes das despesas operacionais
- Se melhorou em relação ao trimestre anterior: +1 ponto

---

## De onde vem cada dado no yfinance

| Critério | Fonte | Disponível |
|----------|-------|--------|
| 1. Net Income | `quarterly_financials` | Sim |
| 2. OCF | `quarterly_cashflow` | Sim (Operating Cash Flow) |
| 3. NI crescente | `quarterly_financials` | Sim |
| 4. OCF > NI | `quarterly_cashflow` + `quarterly_financials` | Sim |
| 5. Ativo crescente | `quarterly_balance_sheet` | Sim (Total Assets) |
| 6. Passivo circulante | `quarterly_balance_sheet` | Sim (Current Liabilities) |
| 7. Liquidez corrente | `quarterly_balance_sheet` | Sim |
| 8. Sem diluição | `quarterly_balance_sheet` | Sim (Ordinary Shares Number) |
| 9. Margem bruta | `quarterly_financials` | Sim |

Os 9 critérios dão para calcular com os dados do yfinance.

In [5]:
# ── Secao 4: Funcao Completa - calculate_piotroski() ──
# 
# A funcao abaixo implementa TODOS os 9 criterios do Piotroski F-Score
# usando dados reais de 3 DataFrames do yfinance:
# - quarterly_financials (P&L)
# - quarterly_balance_sheet (Balanco patrimonial)
# - quarterly_cashflow (Fluxo de caixa)
#

def calculate_piotroski(ticker_financials, ticker_balance_sheet, ticker_cashflow):
    """
    Calcula Piotroski F-Score (0-9) a partir de dados financeiros completos.
    
    Entrada:
        ticker_financials: DataFrame de quarterly_financials do yfinance
        ticker_balance_sheet: DataFrame de quarterly_balance_sheet do yfinance
        ticker_cashflow: DataFrame de quarterly_cashflow do yfinance
    
    Saida:
        dict com 'piotroski_score' (0-9), 'interpretation', e 'details' (dict dos 9 criterios)
    """
    
    score = 0
    details = {}
    
    # Ordena colunas por data (mais recente primeiro)
    fin_df = ticker_financials.sort_index(axis=1, ascending=False)
    bs_df = ticker_balance_sheet.sort_index(axis=1, ascending=False)
    cf_df = ticker_cashflow.sort_index(axis=1, ascending=False)
    
    # Pega dois trimestres: atual (indice 0) e anterior (indice 1)
    current_q = fin_df.iloc[:, 0]
    prior_q = fin_df.iloc[:, 1] if fin_df.shape[1] > 1 else fin_df.iloc[:, 0]
    
    current_bs = bs_df.iloc[:, 0]
    prior_bs = bs_df.iloc[:, 1] if bs_df.shape[1] > 1 else bs_df.iloc[:, 0]
    
    current_cf = cf_df.iloc[:, 0]
    prior_cf = cf_df.iloc[:, 1] if cf_df.shape[1] > 1 else cf_df.iloc[:, 0]
    
    # ── Criterio 1: Net Income > 0 ──
    
    net_income = current_q.get('Net Income', 0)
    
    if net_income > 0:
        score += 1
        details['profitability'] = f"[OK] Net Income positivo (R$ {net_income/1e9:.2f}B)"
    else:
        details['profitability'] = f"[NAO] Net Income negativo ou zero (R$ {net_income/1e9:.2f}B)"
    
    # ── Criterio 2: OCF > 0 ──
    
    ocf = current_cf.get('Operating Cash Flow', 0)
    
    if ocf > 0:
        score += 1
        details['ocf'] = f"[OK] Operating Cash Flow positivo (R$ {ocf/1e9:.2f}B) - lucro com caixa real"
    else:
        details['ocf'] = f"[NAO] Operating Cash Flow negativo ou zero (R$ {ocf/1e9:.2f}B) - lucro artificial/prejuizo"
    
    # ── Criterio 3: Crescimento de Net Income ──
    
    prior_net_income = prior_q.get('Net Income', 0)

    if net_income > prior_net_income:
        score += 1
        details['ni_growth'] = f"[OK] Net Income cresceu ({net_income/1e9:.2f}B vs {prior_net_income/1e9:.2f}B)"
    else:
        details['ni_growth'] = f"[NAO] Net Income diminuiu ({net_income/1e9:.2f}B vs {prior_net_income/1e9:.2f}B)"
    
    # ── Criterio 4: Qualidade de Lucro (OCF > NI?) ──
    
    if ocf > net_income:
        score += 1
        details['cash_quality'] = f"[OK] OCF > Net Income (R$ {ocf/1e9:.2f}B > R$ {net_income/1e9:.2f}B) - lucro tem qualidade"
    else:
        details['cash_quality'] = f"[NAO] OCF <= Net Income (R$ {ocf/1e9:.2f}B <= R$ {net_income/1e9:.2f}B) - lucro pode ser inflado"
    
    # ── Criterio 5: Aumento de Ativos ──
    
    current_assets = current_bs.get('Total Assets', 0)
    prior_assets = prior_bs.get('Total Assets', 0)
    
    if current_assets > 0 and prior_assets > 0:
        if current_assets > prior_assets:
            score += 1
            asset_growth = ((current_assets - prior_assets) / prior_assets) * 100
            details['assets'] = f"[OK] Total Assets cresceu (+{asset_growth:.1f}%: R$ {current_assets/1e9:.2f}B vs R$ {prior_assets/1e9:.2f}B)"
        else:
            asset_decline = ((prior_assets - current_assets) / prior_assets) * 100
            details['assets'] = f"[NAO] Total Assets diminuiu (-{asset_decline:.1f}%: R$ {current_assets/1e9:.2f}B vs R$ {prior_assets/1e9:.2f}B)"
    else:
        details['assets'] = "[ALERTA] Dados de Total Assets incompletos"
    
    # ── Criterio 6: Reducao de Divida ──
    
    current_liabilities = current_bs.get('Current Liabilities', 0)
    prior_liabilities = prior_bs.get('Current Liabilities', 0)
    
    if current_liabilities > 0 and prior_liabilities > 0:
        if current_liabilities < prior_liabilities:
            score += 1
            debt_reduction = ((prior_liabilities - current_liabilities) / prior_liabilities) * 100
            details['debt'] = f"[OK] Divida reduziu (-{debt_reduction:.1f}%: R$ {current_liabilities/1e9:.2f}B vs R$ {prior_liabilities/1e9:.2f}B)"
        else:
            debt_increase = ((current_liabilities - prior_liabilities) / prior_liabilities) * 100
            details['debt'] = f"[NAO] Divida aumentou (+{debt_increase:.1f}%: R$ {current_liabilities/1e9:.2f}B vs R$ {prior_liabilities/1e9:.2f}B)"
    else:
        details['debt'] = "[ALERTA] Dados de Current Liabilities incompletos"
    
    # ── Criterio 7: Aumento de Current Ratio ──
    
    current_ca = current_bs.get('Current Assets', 0)
    prior_ca = prior_bs.get('Current Assets', 0)
    
    if current_ca > 0 and current_liabilities > 0 and prior_ca > 0 and prior_liabilities > 0:
        current_cr = current_ca / current_liabilities
        prior_cr = prior_ca / prior_liabilities
        
        if current_cr > prior_cr:
            score += 1
            details['liquidity'] = f"[OK] Current Ratio melhorou ({current_cr:.2f} vs {prior_cr:.2f}) - melhor liquidez"
        else:
            details['liquidity'] = f"[NAO] Current Ratio piorou ({current_cr:.2f} vs {prior_cr:.2f}) - pior liquidez"
    else:
        details['liquidity'] = "[ALERTA] Dados para Current Ratio incompletos"
    
    # ── Criterio 8: Sem Diluicao de Acoes ──

    current_shares = current_bs.get('Ordinary Shares Number', 0)
    prior_shares = prior_bs.get('Ordinary Shares Number', 0)

    if current_shares > 0 and prior_shares > 0:
        if current_shares <= prior_shares:
            score += 1
            details['shares'] = f"[OK] Sem diluicao ({current_shares/1e9:.2f} vs {prior_shares/1e9:.2f} bilhoes de acoes ordinarias)"
        else:
            dilucao_pct = ((current_shares - prior_shares) / prior_shares) * 100
            details['shares'] = f"[NAO] Diluicao detectada (+{dilucao_pct:.1f}% novas acoes ordinarias) - seu % da empresa caiu"
    else:
        details['shares'] = "[ALERTA] Dados de shares ordinarias nao disponivel neste trimestre"
    
    # ── Criterio 9: Aumento de Gross Margin ──
    
    current_revenue = current_q.get('Total Revenue', 1)
    current_cogs = current_q.get('Cost Of Revenue', 0)
    current_gm = (current_revenue - current_cogs) / current_revenue if current_revenue > 0 else 0
    
    prior_revenue = prior_q.get('Total Revenue', 1)
    prior_cogs = prior_q.get('Cost Of Revenue', 0)
    prior_gm = (prior_revenue - prior_cogs) / prior_revenue if prior_revenue > 0 else 0
    
    if current_gm > prior_gm:
        score += 1
        details['margin'] = f"[OK] Gross Margin melhorou ({current_gm:.1%} vs {prior_gm:.1%})"
    else:
        details['margin'] = f"[NAO] Gross Margin caiu ({current_gm:.1%} vs {prior_gm:.1%})"
    
    # ── Retorno ──
    
    # Interpretacao do score final
    if score >= 8:
        interpretation = 'Excelente (8-9) - Empresa com fundamentals muito solidos'
    elif score >= 6:
        interpretation = 'Bom (6-7) - Empresa competente, algumas fraquezas'
    else:
        interpretation = 'Fraco (0-5) - Empresa com serios problemas financeiros'
    
    return {
        'piotroski_score': score,
        'interpretation': interpretation,
        'details': details
    }


print("OK Funcao calculate_piotroski COMPLETA e FUNCIONAL!")
print("OK Todos os 9 criterios implementados com dados reais")
print("OK Proxima celula: teste com PETR4")

OK Funcao calculate_piotroski COMPLETA e FUNCIONAL!
OK Todos os 9 criterios implementados com dados reais
OK Proxima celula: teste com PETR4


In [6]:
# ── Secao 5: Teste de execucao ──

# Busca dados financeiros completos de PETR4
ticker = yf.Ticker("PETR4.SA")
df_inc = ticker.quarterly_financials
df_bs = ticker.quarterly_balance_sheet
df_cf = ticker.quarterly_cashflow

# Roda a funcao com todos os 3 DataFrames
result = calculate_piotroski(df_inc, df_bs, df_cf)

print(f"\n{'='*70}")
print(f"  Piotroski F-Score: {result['piotroski_score']}/9")
print(f"  Interpretacao: {result['interpretation']}")
print(f"{'='*70}\n")

print("Detalhes dos 9 criterios:\n")
for i, (criterion, status) in enumerate(result['details'].items(), 1):
    print(f"  {i}. {status}\n")


  Piotroski F-Score: 9/9
  Interpretacao: Excelente (8-9) - Empresa com fundamentals muito solidos

Detalhes dos 9 criterios:

  1. [OK] Net Income positivo (R$ 10.43B)

  2. [OK] Operating Cash Flow positivo (R$ 12.25B) - lucro com caixa real

  3. [OK] Net Income cresceu (10.43B vs 6.20B)

  4. [OK] OCF > Net Income (R$ 12.25B > R$ 10.43B) - lucro tem qualidade

  5. [OK] Total Assets cresceu (+4.4%: R$ 247.08B vs R$ 236.58B)

  6. [OK] Divida reduziu (-2.4%: R$ 35.04B vs R$ 35.92B)

  7. [OK] Current Ratio melhorou (0.85 vs 0.74) - melhor liquidez

  8. [OK] Sem diluicao (12.89 vs 12.89 bilhoes de acoes ordinarias)

  9. [OK] Gross Margin melhorou (58.0% vs 48.2%)



## O que virou no projeto

O protótipo acima calcula o Piotroski com dados **trimestrais**. No projeto, a conta virou a tool
`calcular_piotroski_score` (`fluxo_financeiro/tools/financial_data.py`), com três mudanças:

- usa os demonstrativos **anuais** (`financials`, `balance_sheet`, `cashflow`), comparando o último ano com o anterior: trimestre a trimestre, a sazonalidade fazia critérios oscilarem sem a empresa ter mudado;
- é uma **adaptação** do Piotroski clássico, que usa ROA, dívida de longo prazo e giro do ativo: aqui entram os campos que o yfinance entrega (lucro, caixa operacional, ativo total, passivo circulante, liquidez corrente, ações em circulação e margem bruta);
- as faixas mudaram para **7 a 9 excelente, 4 a 6 bom e 0 a 3 fraco**.

A lógica fica em `_calcular_score()`, uma função pura, que também é usada pelo detector de anomalias (notebook 10: Piotroski de 3 ou menos é sinal de anomalia).

In [7]:
from fluxo_financeiro.tools.financial_data import calcular_piotroski_score

print(calcular_piotroski_score.invoke("PETR4"))

2026-11-06 21:37:40,113 - fluxo_financeiro - INFO - [tool] calcular_piotroski_score: ticker=PETR4.SA


2026-11-06 21:37:41,184 - fluxo_financeiro - INFO - [tool] calcular_piotroski_score: ok — score=7/9


Piotroski F-Score para PETR4.SA: 7/9 — Excelente (7-9) - Empresa com fundamentals muito solidos

Critérios:
  [OK] Net Income positivo (R$ 19.63B)
  [OK] Operating Cash Flow positivo (R$ 36.05B)
  [OK] Net Income cresceu (19.63B vs 7.53B)
  [OK] OCF > Net Income — lucro com qualidade de caixa
  [OK] Total Assets cresceu (+22.4%)
  [NAO] Divida corrente aumentou (+14.6%)
  [OK] Current Ratio melhorou (0.71 vs 0.69)
  [OK] Sem diluicao de acoes
  [NAO] Gross Margin caiu (47.6% vs 50.3%)



## O que observar

- O score da tool pode diferir do protótipo: anual × trimestral muda a comparação.
- A saída lista cada critério com `[OK]` ou `[NAO]`: é esse texto que o agente YFinance repassa.